In [4]:
#!pip install imblearn

import pandas as pd
import numpy as np

# Sklearn imports for modeling, evaluation, and tuning
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (roc_auc_score, recall_score, f1_score, 
                             accuracy_score, precision_score, average_precision_score)

# Imblearn imports for SMOTE and safe pipelining
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# 1. Load the data
# Make sure "scaled_data.csv" is in the same directory as your notebook
df = pd.read_csv("./data/cleaned_data/scaled_data.csv")
X = df.drop(columns=['Bankrupt?'])
y = df['Bankrupt?']

# 2. Train/Test Split
# Stratify to ensure the 20% test set has the exact same ratio of bankruptcies
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Define the Pipeline and apply SMOTE before modeling
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('knn', KNeighborsClassifier())
])

# 4. Define the parameters to be tuned and their range which will be iterated over using Grid Search
param_grid = {
    'feature_selection__k': [10, 15, 20, 30],       # Number of top columns to use
    'knn__n_neighbors': [3, 5, 7, 11, 15],          # Tuning k-neighbors
    'knn__weights': ['uniform', 'distance'],        # Tuning weight types
    'knn__metric': ['euclidean', 'manhattan']       # Tuning distance measurement
}

# 5. Execute Grid Search for Best Model
print("Running model optimization...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use ROC-AUC to select the best model
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=cv, 
    scoring='roc_auc', 
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

# 6. Evaluate the Best Model on the Untouched Test Set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# 7. Calculate Performance Metrics Decided on by Team
metrics_dict = {
    "Metric": [
        "ROC-AUC", 
        "PR-AUC (Precision-Recall AUC)", 
        "Recall (Class 1)", 
        "Precision (Class 1)", 
        "F1-Score (Class 1)", 
        "Accuracy"
    ],
    "Score": [
        roc_auc_score(y_test, y_pred_proba),
        average_precision_score(y_test, y_pred_proba),  # PR-AUC
        recall_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        accuracy_score(y_test, y_pred)
    ]
}

# 8. Output as a DataFrame (Table)
print("\n--- OPTIMIZATION RESULTS ---")
print(f"Best Parameters Found:\n{grid_search.best_params_}\n")

results_df = pd.DataFrame(metrics_dict)
results_df['Score'] = results_df['Score'].round(4) # Round for readability

print("--- FINAL MODEL PERFORMANCE ON TEST SET ---")
print(f"Model Description: Optimal KNN Model ({grid_search.best_params_['knn__n_neighbors']} Neighbors, {grid_search.best_params_['feature_selection__k']} Features) trained on SMOTE data.")
display(results_df) # 'display' outputs a cleanly formatted HTML table in Jupyter

Running optimizations. This may take a minute or two...


/usr/local/lib/python3.12/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,



--- OPTIMIZATION RESULTS ---
Best Parameters Found:
{'feature_selection__k': 10, 'knn__metric': 'manhattan', 'knn__n_neighbors': 15, 'knn__weights': 'uniform'}

--- FINAL MODEL PERFORMANCE ON TEST SET ---
Model Description: Optimal KNN Model (15 Neighbors, 10 Features) trained on SMOTE data.


,Metric,Score
0,ROC-AUC,0.8944
1,PR-AUC (Precision-Recall AUC),0.2811
2,Recall (Class 1),0.7955
3,Precision (Class 1),0.1659
4,F1-Score (Class 1),0.2745
5,Accuracy,0.8644
